```{contents}
```

## Exploding Gradients

### Concept Overview

**Exploding gradients** occur when gradients grow exponentially during backpropagation, causing parameter updates to become extremely large. This destabilizes training and prevents convergence.

Formally, for deep networks:

$$
\frac{\partial \mathcal{L}}{\partial W} = \prod_{i=1}^{L} J_i
$$

If the Jacobian matrices (J_i) have eigenvalues > 1, the product grows exponentially with depth (L).

---

### Intuition

Consider passing a number through many multiplications:

```
1.2 × 1.2 × 1.2 × ... × 1.2 → very large
```

Similarly, during backpropagation:

* Each layer multiplies the gradient.
* If multipliers > 1 → gradient explodes.
* Leads to:

  * NaN weights
  * Loss becomes infinite
  * Training collapses

Occurs frequently in:

* Very deep feedforward networks
* Recurrent Neural Networks (RNNs)
* Transformers with poor initialization

---

### Symptoms

| Symptom   | Description                 |
| --------- | --------------------------- |
| Loss      | Suddenly becomes NaN or Inf |
| Weights   | Grow uncontrollably         |
| Gradients | Extremely large norms       |
| Training  | Unstable or diverging       |

---

### Why It Happens

| Cause                      | Effect                         |
| -------------------------- | ------------------------------ |
| Poor weight initialization | Large activations              |
| Deep network depth         | Multiplicative gradient growth |
| High learning rate         | Overshoots minimum             |
| Unbounded activations      | No normalization               |

---

### Mathematical View

Backpropagation:

$$
\delta_l = (W_{l+1}^T \delta_{l+1}) \odot f'(z_l)
$$

Repeated multiplication by large weights causes (\delta_l) to explode.

---

### Training Workflow With Exploding Gradients

```
Forward Pass
    ↓
Compute Loss
    ↓
Backward Pass
    ↓
Gradients explode
    ↓
Weights become NaN
    ↓
Training diverges
```

---

### Remediation Techniques

| Method                | Purpose                 |
| --------------------- | ----------------------- |
| Gradient Clipping     | Cap gradient magnitude  |
| Weight Initialization | Xavier / He             |
| Normalization         | BatchNorm / LayerNorm   |
| Activation Functions  | ReLU, GELU              |
| Smaller Learning Rate | Reduce update size      |
| Residual Connections  | Stabilize deep networks |

---

### Gradient Clipping (Primary Fix)

$$
g \leftarrow \frac{g}{\max(1, \frac{||g||}{\tau})}
$$

---

### PyTorch Demonstration

#### Model That Explodes



In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

model = nn.Sequential(
    *[nn.Linear(100, 100) for _ in range(20)],
    nn.Linear(100, 1)
)

optimizer = optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

x = torch.randn(64, 100)
y = torch.randn(64, 1)

for step in range(50):
    optimizer.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()

    total_norm = torch.norm(torch.stack([p.grad.norm() for p in model.parameters()]))
    print(f"Step {step} | Grad norm: {total_norm:.2f}")

    optimizer.step()


Step 0 | Grad norm: 0.68
Step 1 | Grad norm: 0.40
Step 2 | Grad norm: 0.24
Step 3 | Grad norm: 0.14
Step 4 | Grad norm: 0.08
Step 5 | Grad norm: 0.05
Step 6 | Grad norm: 0.03
Step 7 | Grad norm: 0.02
Step 8 | Grad norm: 0.01
Step 9 | Grad norm: 0.01
Step 10 | Grad norm: 0.00
Step 11 | Grad norm: 0.00
Step 12 | Grad norm: 0.00
Step 13 | Grad norm: 0.00
Step 14 | Grad norm: 0.00
Step 15 | Grad norm: 0.00
Step 16 | Grad norm: 0.00
Step 17 | Grad norm: 0.00
Step 18 | Grad norm: 0.00
Step 19 | Grad norm: 0.00
Step 20 | Grad norm: 0.00
Step 21 | Grad norm: 0.00
Step 22 | Grad norm: 0.00
Step 23 | Grad norm: 0.00
Step 24 | Grad norm: 0.00
Step 25 | Grad norm: 0.00
Step 26 | Grad norm: 0.00
Step 27 | Grad norm: 0.00
Step 28 | Grad norm: 0.00
Step 29 | Grad norm: 0.00
Step 30 | Grad norm: 0.00
Step 31 | Grad norm: 0.00
Step 32 | Grad norm: 0.00
Step 33 | Grad norm: 0.00
Step 34 | Grad norm: 0.00
Step 35 | Grad norm: 0.00
Step 36 | Grad norm: 0.00
Step 37 | Grad norm: 0.00
Step 38 | Grad norm: 0



Gradients quickly become extremely large.

---

### Apply Gradient Clipping



In [2]:
for step in range(50):
    optimizer.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()




Now training remains stable.

---

### Advanced Variants

| Variant               | Where Used             |
| --------------------- | ---------------------- |
| Global Norm Clipping  | RNNs, Transformers     |
| Adaptive Clipping     | Reinforcement Learning |
| Layer-wise Clipping   | Very deep models       |
| Norm-based Scheduling | Large-scale training   |

---

### Architectural Solutions

| Architecture | Benefit                  |
| ------------ | ------------------------ |
| ResNet       | Gradient highways        |
| LSTM / GRU   | Gated gradient flow      |
| Transformer  | Residual + normalization |

---

### Summary

| Aspect                  | Key Insight                              |
| ----------------------- | ---------------------------------------- |
| Root cause              | Exponential gradient growth              |
| Primary symptom         | NaN loss                                 |
| Core fix                | Gradient clipping                        |
| Supporting fixes        | Initialization, normalization, residuals |
| Deep learning relevance | Critical for deep & recurrent models     |

---

### Practical Rule

If training suddenly diverges with NaNs, **check gradient norms first**.

